In [1]:
import pandas as pd
import os

pd.set_option('display.max_columns', None) 

In [2]:
folder = "../data/"

def load(name):
    return pd.read_csv(os.path.join(folder, name))

df1 = load("armed-forces-personnel.csv")                
df2 = load("civilian-and-combatant-deaths-in-armed-conflicts.csv")
df3 = load("deaths-in-armed-conflicts-by-region.csv")
df4 = load("military-spending-as-a-share-of-gdp-sipri.csv")
df5 = load("percentage-of-territory-controlled-by-government.csv")
df6 = load("political-regime.csv")
df7 = load("rigorous-and-impartial-public-administration-score.csv")
dfs = [df1, df2, df3, df4, df5, df6, df7]

YEARS = list(range(1989, 2025))       
def clean_df(df):
    df['Entity'] = df['Entity'].str.strip()
    df = df[df['Year'].isin(YEARS)]
    df = df.drop_duplicates(subset=['Entity', 'Year'])
    return df

dfs = [clean_df(df) for df in dfs]


key_cols = ['Entity','Code', 'Year']
for i, df in enumerate(dfs, start=1):
    rename_dict = {col: f"df{i}_{col}" for col in df.columns if col not in key_cols}
    dfs[i-1] = df.rename(columns=rename_dict)
from functools import reduce
merged = reduce(lambda left, right: pd.merge(left, right, on=['Entity','Code','Year'], how='outer'), dfs)

print(merged.columns)

Index(['Entity', 'Code', 'Year', 'df1_Armed forces personnel, total',
       'df1_World regions according to OWID',
       'df2_Deaths of civilians in ongoing conflicts - Conflict type: all',
       'df2_Deaths of unknown type in ongoing conflicts - Conflict type: all',
       'df2_Deaths of combatants in ongoing conflicts - Conflict type: all',
       'df3_Deaths in ongoing conflicts (best estimate) - Conflict type: all',
       'df4_Military expenditure (% of GDP)',
       'df4_World regions according to OWID',
       'df5_Territory under state control (central estimate)',
       'df5_World regions according to OWID', 'df6_Political regime',
       'df7_Rigorous and impartial public administration (central estimate)'],
      dtype='object')


In [10]:
pct_nan = (merged.isna().sum() / len(merged) * 100).sort_values(ascending=False)
print(pct_nan.round(2))

df4_Military expenditure (% of GDP)                                     24.99
df1_Armed forces personnel, total                                       24.63
df5_Territory under state control (central estimate)                    11.20
df6_Political regime                                                    10.96
df7_Rigorous and impartial public administration (central estimate)     10.74
df3_Deaths in ongoing conflicts (best estimate) - Conflict type: all     0.03
df2_Deaths of unknown type in ongoing conflicts - Conflict type: all     0.03
df2_Deaths of combatants in ongoing conflicts - Conflict type: all       0.03
df2_Deaths of civilians in ongoing conflicts - Conflict type: all        0.03
Year                                                                     0.00
Entity                                                                   0.00
Code                                                                     0.00
dtype: float64


In [4]:
merged['Entity'].unique()

array(['Abkhazia', 'Abyei', 'Afghanistan', 'Africa',
       'Africa (population-weighted)', 'Aland Islands', 'Albania',
       'Algeria', 'American Samoa', 'Americas', 'Andorra', 'Angola',
       'Anguilla', 'Antigua and Barbuda', 'Argentina', 'Armenia', 'Aruba',
       'Asia', 'Asia (population-weighted)', 'Asia and Oceania',
       'Australia', 'Austria', 'Austria-Hungary', 'Azerbaijan', 'Bahamas',
       'Bahrain', 'Bangladesh', 'Barbados', 'Belarus', 'Belgium',
       'Belize', 'Benin', 'Bermuda', 'Bhutan', 'Bolivia',
       'Bonaire Sint Eustatius and Saba', 'Bosnia and Herzegovina',
       'Botswana', 'Brazil', 'British Indian Ocean Territory',
       'British Virgin Islands', 'Brunei', 'Bulgaria', 'Burkina Faso',
       'Burundi', 'Cambodia', 'Cameroon', 'Canada', 'Cape Verde',
       'Cayman Islands', 'Central African Republic', 'Chad', 'Chile',
       'China', 'Christmas Island', 'Cocos Islands', 'Colombia',
       'Comoros', 'Congo', 'Cook Islands', 'Costa Rica', "Cote d'Ivoi

In [5]:
drop_cols = ['df1_World regions according to OWID',
            'df5_World regions according to OWID',
            'df4_World regions according to OWID']
merged.drop(columns=drop_cols, axis=1, inplace=True)
merged.dropna(subset=['Code'], inplace=True)


In [6]:
ONU = {
    'Afghanistan', 'Albania', 'Algeria', 'Andorra', 'Angola',
    'Antigua and Barbuda', 'Argentina', 'Armenia', 'Australia', 'Austria',
    'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barbados',
    'Belarus', 'Belgium', 'Belize', 'Benin', 'Bhutan', 'Bolivia',
    'Bosnia and Herzegovina', 'Botswana', 'Brazil', 'Brunei', 'Bulgaria',
    'Burkina Faso', 'Burundi', 'Cabo Verde', 'Cambodia', 'Cameroon',
    'Canada', 'Central African Republic', 'Chad', 'Chile', 'China',
    'Colombia', 'Comoros', 'Congo', 'Costa Rica', "Cote d'Ivoire",
    'Croatia', 'Cuba', 'Cyprus', 'Czechia', 'Democratic Republic of Congo',
    'Denmark', 'Djibouti', 'Dominica', 'Dominican Republic', 'Ecuador',
    'Egypt', 'El Salvador', 'Equatorial Guinea', 'Eritrea', 'Estonia',
    'Eswatini', 'Ethiopia', 'Fiji', 'Finland', 'France', 'Gabon',
    'Gambia', 'Georgia', 'Germany', 'Ghana', 'Greece', 'Grenada',
    'Guatemala', 'Guinea', 'Guinea-Bissau', 'Guyana', 'Haiti', 'Honduras',
    'Hungary', 'Iceland', 'India', 'Indonesia', 'Iran', 'Iraq', 'Ireland',
    'Israel', 'Italy', 'Jamaica', 'Japan', 'Jordan', 'Kazakhstan',
    'Kenya', 'Kiribati', 'Kuwait', 'Kyrgyzstan', 'Laos', 'Latvia',
    'Lebanon', 'Lesotho', 'Liberia', 'Libya', 'Liechtenstein', 'Lithuania',
    'Luxembourg', 'Madagascar', 'Malawi', 'Malaysia', 'Maldives', 'Mali',
    'Malta', 'Marshall Islands', 'Mauritania', 'Mauritius', 'Mexico',
    'Micronesia (country)', 'Moldova', 'Monaco', 'Mongolia', 'Montenegro',
    'Morocco', 'Mozambique', 'Myanmar', 'Namibia', 'Nauru', 'Nepal',
    'Netherlands', 'New Zealand', 'Nicaragua', 'Niger', 'Nigeria',
    'North Korea', 'North Macedonia', 'Norway', 'Oman', 'Pakistan',
    'Palau', 'Panama', 'Papua New Guinea', 'Paraguay', 'Peru',
    'Philippines', 'Poland', 'Portugal', 'Qatar', 'Romania', 'Russia',
    'Rwanda', 'Saint Kitts and Nevis', 'Saint Lucia',
    'Saint Vincent and the Grenadines', 'Samoa', 'San Marino',
    'Sao Tome and Principe', 'Saudi Arabia', 'Senegal', 'Serbia',
    'Seychelles', 'Sierra Leone', 'Singapore', 'Slovakia', 'Slovenia',
    'Solomon Islands', 'Somalia', 'South Africa', 'South Korea',
    'South Sudan', 'Spain', 'Sri Lanka', 'Sudan', 'Suriname', 'Sweden',
    'Switzerland', 'Syria', 'Tajikistan', 'Tanzania', 'Thailand',
    'Timor-Leste', 'Togo', 'Tonga', 'Trinidad and Tobago', 'Tunisia',
    'Turkey', 'Turkmenistan', 'Tuvalu', 'Uganda', 'Ukraine',
    'United Arab Emirates', 'United Kingdom', 'United States', 'Uruguay',
    'Uzbekistan', 'Vanuatu', 'Vatican', 'Venezuela', 'Vietnam', 'Yemen',
    'Zambia', 'Zimbabwe'
}

# 2. Filtrar
merged = merged[merged['Entity'].isin(ONU)].copy()

In [7]:
merged.info

<bound method DataFrame.info of            Entity Code  Year  df1_Armed forces personnel, total  \
72    Afghanistan  AFG  1989                            55000.0   
73    Afghanistan  AFG  1990                            58000.0   
74    Afghanistan  AFG  1991                            45000.0   
75    Afghanistan  AFG  1992                            45000.0   
76    Afghanistan  AFG  1993                            45000.0   
...           ...  ...   ...                                ...   
8636     Zimbabwe  ZWE  2020                            51000.0   
8637     Zimbabwe  ZWE  2021                                NaN   
8638     Zimbabwe  ZWE  2022                                NaN   
8639     Zimbabwe  ZWE  2023                                NaN   
8640     Zimbabwe  ZWE  2024                                NaN   

      df2_Deaths of civilians in ongoing conflicts - Conflict type: all  \
72                                                303.0                   
73           

In [8]:
pct_nan = (merged.isna().sum() / len(merged) * 100).sort_values(ascending=False)
print(pct_nan.round(2))


df4_Military expenditure (% of GDP)                                     24.99
df1_Armed forces personnel, total                                       24.63
df5_Territory under state control (central estimate)                    11.20
df6_Political regime                                                    10.96
df7_Rigorous and impartial public administration (central estimate)     10.74
df3_Deaths in ongoing conflicts (best estimate) - Conflict type: all     0.03
df2_Deaths of unknown type in ongoing conflicts - Conflict type: all     0.03
df2_Deaths of combatants in ongoing conflicts - Conflict type: all       0.03
df2_Deaths of civilians in ongoing conflicts - Conflict type: all        0.03
Year                                                                     0.00
Entity                                                                   0.00
Code                                                                     0.00
dtype: float64


In [9]:
print(merged.head())

         Entity Code  Year  df1_Armed forces personnel, total  \
72  Afghanistan  AFG  1989                            55000.0   
73  Afghanistan  AFG  1990                            58000.0   
74  Afghanistan  AFG  1991                            45000.0   
75  Afghanistan  AFG  1992                            45000.0   
76  Afghanistan  AFG  1993                            45000.0   

    df2_Deaths of civilians in ongoing conflicts - Conflict type: all  \
72                                              303.0                   
73                                              101.0                   
74                                               49.0                   
75                                             1684.0                   
76                                              637.0                   

    df2_Deaths of unknown type in ongoing conflicts - Conflict type: all  \
72                                             4043.0                      
73                

In [ ]:
merged.to_csv('')